In [ ]:
# lemmatize code Hebrew text

In [ ]:
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('dicta-il/dictabert-lex')
model = AutoModel.from_pretrained('dicta-il/dictabert-lex', trust_remote_code=True)

model.eval()

sentence = 'בשנת 1948 השלים אפרים קישון את לימודיו בפיסול מתכת ובתולדות האמנות והחל לפרסם מאמרים הומוריסטיים'
print(model.predict([sentence], tokenizer))

config.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

BertForLexPrediction.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/dicta-il/dictabert-lex:
- BertForLexPrediction.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[[('בשנת', 'שנה'), ('1948', '1948'), ('השלים', 'השלים'), ('אפרים', 'אפרים'), ('קישון', 'קישון'), ('את', 'את'), ('לימודיו', 'לימוד'), ('בפיסול', 'פיסול'), ('מתכת', 'מתכת'), ('ובתולדות', 'תולדה'), ('האמנות', 'אומנות'), ('והחל', 'החל'), ('לפרסם', 'פרסם'), ('מאמרים', 'מאמר'), ('הומוריסטיים', 'הומוריסטי')]]


In [ ]:
import pandas as pd

In [ ]:
verses = pd.read_csv('all_verses_sims.csv')

In [ ]:
result = model.predict(['אשרי־האיש אשר לא הלך בעצת רשעים ובדרך חטאים לא עמד ובמושב לצים לא ישב'], tokenizer)

In [ ]:
result[0][0]

('אשרי', 'אשרי')

In [ ]:
import string

In [ ]:
trans_table = str.maketrans('', '', string.punctuation)

In [ ]:
verse = verses['Hebrew'].iloc[0]
verse = verse.replace('־', ' ')
verse

'אשרי האיש אשר לא הלך בעצת רשעים ובדרך חטאים לא עמד ובמושב לצים לא ישב'

In [ ]:
result = model.predict([verse], tokenizer)[0]

In [ ]:
lemma_occurrences = {}
blank_word_occurences = {}
verses_results = []

for verse in verses['Hebrew'].iloc:
  verse = verse.replace('־', ' ')
  verse = verse.translate(trans_table)
  result = model.predict([verse], tokenizer)[0]

  curr_verse_result = ''
  for i in result:
    lemma = i[1]
    if lemma == '[BLANK]':
      word = i[0]
      curr_verse_result += word + ' '
      if word in blank_word_occurences:
        blank_word_occurences[word] += 1
      else:
        blank_word_occurences[word] = 1
    else:
      curr_verse_result += lemma + ' '
      if lemma in lemma_occurrences:
        lemma_occurrences[lemma] +=1
      else:
        lemma_occurrences[lemma] = 1

  verses_results.append(curr_verse_result[:-1])


In [ ]:
len(verses_results)

2527

In [ ]:
psalm_nums = verses[['Psalm', 'Verse']].copy()

In [ ]:
psalm_nums['lemmatized_verse'] = verses_results

In [ ]:
psalm_nums.to_csv('hebrew_lemmatized_verses.csv')

In [ ]:
hebrew_lemma_counts = pd.DataFrame({'lemma' : lemma_occurrences.keys(), 'count':lemma_occurrences.values()})

In [ ]:
hebrew_stop_words = ['כי', 'כל', 'לא', 'על', 'את', 'לי', 'ולא', 'אשר',
                     'אתה', 'עד', 'עלי', 'לך', 'עם', 'אם', 'אליך', 'להם', 'למען',
                     'מאד', 'ואני', 'ממני', 'אל', 'לו', 'בכל', 'מה', 'מי', 'הוא', 'אני', 'אין', 'ואל', 'בך',
                     'כול', 'היי']

In [ ]:
non_stop_hebrew = hebrew_lemma_counts[~hebrew_lemma_counts['lemma'].isin(hebrew_stop_words)]

In [ ]:
non_stop_hebrew.sort_values('count', ascending = False).head(30)

,lemma,count
15,יהוה,662
115,אלוהים,377
49,ארץ,193
221,שם,155
111,נפש,144
18,יום,125
193,חסד,123
74,בן,105
166,לב,101
218,לעולם,94


In [ ]:
hebrew_lemma_counts.sort_values('count', ascending = False).to_csv('hebrew_summary_counts.csv')

In [ ]:
hebrew_unknown_counts = pd.DataFrame({'lemma' : blank_word_occurences.keys(), 'count':blank_word_occurences.values() })

In [ ]:
hebrew_unknown_counts.sort_values('count', ascending = False).to_csv('hebrew_unnassigned_lemmas.csv')

In [ ]:
hebrew_unknown_counts.to_csv('hebrew_unnassigned_lemmas.csv')